# 02 — Preprocessing: Taiwan Credit Card Default

## 1. Imports and Paths

In [ ]:
import pandas as pd
import numpy as np
import os

RAW_PATH = '../data/raw/taiwan/default of credit card clients.xls'
OUT_PATH = '../data/processed/v4/taiwan_credit.csv'

os.makedirs('../data/processed/v4', exist_ok=True)

## 2. Load

In [ ]:
df = pd.read_excel(RAW_PATH, header=1)

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 3. Drop ID and Rename Target

In [ ]:
df = df.drop(columns=['ID'])

df = df.rename(columns={'default payment next month': 'target'})

print(f'Shape after ID drop: {df.shape}')
print(f'Target: {df["target"].value_counts().to_dict()}')
print(f'Default rate: {df["target"].mean():.4f}')

## 4. EDUCATION — Clean Undefined Codes

In [ ]:
print('EDUCATION before cleaning:')
print(df['EDUCATION'].value_counts().sort_index())

df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})

print('\nEDUCATION after cleaning:')
print(df['EDUCATION'].value_counts().sort_index())


## 5. MARRIAGE — Clean Undefined Code

In [ ]:
print('MARRIAGE before cleaning:')
print(df['MARRIAGE'].value_counts().sort_index())

df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

print('\nMARRIAGE after cleaning:')
print(df['MARRIAGE'].value_counts().sort_index())


## 6. SEX — Binary Recode

In [ ]:
df['SEX'] = (df['SEX'] == 1).astype(int)

print('SEX after recode (1=male, 0=female):')
print(df['SEX'].value_counts())

## 7. PAY_* Columns — Check

In [ ]:
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

print('PAY_0 value distribution (representative of all PAY_* columns):')
print(df['PAY_0'].value_counts().sort_index())
print(f'\nRange PAY_0: {df["PAY_0"].min()} bis {df["PAY_0"].max()}')

print('\nNegative BILL_AMT (overpayment, valid):')
bill_cols = [f'BILL_AMT{i}' for i in range(1, 7)]
neg_bills = {c: (df[c] < 0).sum() for c in bill_cols}
print(neg_bills)

## 8. Sanity Checks

In [ ]:
missing = df.isnull().sum()
print('Missing values:')
print(missing[missing > 0] if missing.any() else 'Keine — sauber.')

print('\nDtype overview:')
print(df.dtypes.value_counts())

print(f'\nFinaler Shape: {df.shape}')
print(f'Default rate: {df["target"].mean():.4f}  ({df["target"].sum()} defaults of {len(df)})')

In [ ]:
num_cols = ['LIMIT_BAL', 'AGE'] + [f'BILL_AMT{i}' for i in range(1,7)] + [f'PAY_AMT{i}' for i in range(1,7)]
df[num_cols].describe().round(1)

## 9. Save

In [ ]:
df.insert(0, 'source_row_id', np.arange(len(df), dtype=np.int64))
cols = [c for c in df.columns if c != 'target'] + ['target']
df = df[cols]

df.to_csv(OUT_PATH, index=False)
print(f'Gespeichert: {OUT_PATH}')
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
import sys
sys.path.insert(0, '..')
from src.semiraw import audit_semiraw_export

report = audit_semiraw_export(OUT_PATH, 'taiwan', expect_missing=False)